# Household-grouped any-wasting prediction: XGBoost and LightGBM

This corrected notebook compares XGBoost and LightGBM under two strategies: **SMOTENC without child sample weights** and **child sample weights without SMOTENC**.

**Sampling design:** children sharing a household must never be split across training and validation/test sets. A composite group is formed from `division` + `hh1`; all splits use `StratifiedGroupKFold`. Group IDs are numeric factor codes built from the pair, not concatenated strings, avoiding formatting/collision problems.

**Missing data:** MICE (`miceforest`) is fitted only on each training fold and used to impute that fold's validation data. Only columns with training-fold missingness are modeled; validation-only gaps use training-only median/mode fallback. Non-finite values are sanitized, and a KDTree/PMM failure triggers a fresh MICE run with donor matching disabled.

**Preprocessing order:** MICE → SMOTENC (when selected) → OneHotEncoder. Categorical variables remain categorical during SMOTENC.

**Important:** the notebook is committed as source code, not as executed results. Run it top-to-bottom with `MICS-child-ISRC-revised.dta`. The notebook does not fabricate performance numbers.

**Compatibility fix:** use `lightgbm==4.5.0` with `miceforest==6.0.5`. The previous LightGBM private-method monkey patch has been removed; it could itself trigger a positional/keyword argument mismatch.

## 1. Install dependencies

In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

# Install the tested compatibility pair before importing either package.
REQUIRED = ["lightgbm==4.5.0", "miceforest==6.0.5"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *REQUIRED])

for package, expected in [("lightgbm", "4.5.0"), ("miceforest", "6.0.5")]:
    actual = version(package)
    if actual != expected:
        raise RuntimeError(
            f"{package}=={expected} is required, but {actual} is active. "
            "Restart the notebook kernel and run all cells again."
        )
print("Compatible LightGBM/miceforest versions installed. If either was already imported, restart the kernel now.");

## 2. Imports and reproducibility

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import miceforest as mf
import shap

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay,
    precision_recall_curve
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
CV_FOLDS = 5
MICE_ITERATIONS = 5
np.random.seed(SEED)

## 3. Load the revised MICS child dataset

In [ ]:
DATA_PATH = Path("MICS-child-ISRC-revised.dta")
if not DATA_PATH.exists():
    candidates = [p for p in [Path("MICS-child-ISRC.dta"), *Path(".").glob("*.dta")] if p.exists()]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Set DATA_PATH explicitly; found: {candidates}")
    DATA_PATH = candidates[0]

df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Loaded:", DATA_PATH, "| shape:", df.shape)
print("Weight-like fields:", [c for c in df.columns if "weight" in c.lower()])

## 4. Define outcome, predictors, weights and household groups

The household identifier is formed by combining the division code with `hh1`. This avoids treating the same household number in different divisions as the same household. The identifier is used only for splitting, never as a predictor.

Rows are excluded only if the outcome, positive child weight, or household grouping keys are unavailable. Missing predictor values are retained for MICE.

In [ ]:
TARGET = "any_wasting"
PREDICTORS = [
    "age_m","female","division","urban","wb4","parity_cat","diarrhoea_2w",
    "cough_2w","breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level",
    "electricity","wealth","birth_order","hh_size","water_treated",
    "water_boil_filter","handwash_place","handwash_water","floor_quality","ari_chest"
]
CATEGORICAL_COLUMNS = [
    "female","division","urban","parity_cat","diarrhoea_2w","cough_2w",
    "breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level","electricity",
    "wealth","birth_order","water_treated","water_boil_filter","handwash_place",
    "handwash_water","floor_quality","ari_chest"
]
NUMERIC_COLUMNS = [c for c in PREDICTORS if c not in CATEGORICAL_COLUMNS]
WEIGHT_CANDIDATES = ["ch_weight", "chweight"]
WEIGHT_COL = next((c for c in WEIGHT_CANDIDATES if c in df.columns), None)

required = list(dict.fromkeys([TARGET] + PREDICTORS + ["hh1", "division"]))
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")
if WEIGHT_COL is None:
    raise KeyError("Child sample weight not found: expected ch_weight or chweight.")

model_df = df[list(dict.fromkeys(required + [WEIGHT_COL]))].copy()
model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="coerce")
model_df[WEIGHT_COL] = pd.to_numeric(model_df[WEIGHT_COL], errors="coerce")
model_df = model_df.dropna(subset=[TARGET, WEIGHT_COL, "hh1", "division"])
model_df = model_df.loc[model_df[WEIGHT_COL] > 0].copy()
model_df[TARGET] = model_df[TARGET].astype(int)
if not set(model_df[TARGET].unique()).issubset({0, 1}) or model_df[TARGET].nunique() != 2:
    raise ValueError("any_wasting must contain both binary classes coded 0 and 1.")

# Encode the (division, hh1) pair as an integer group label. This is a grouping key only.
# Factorizing a MultiIndex avoids ambiguous string concatenation and preserves exact pairs.
household_pairs = pd.MultiIndex.from_frame(
    model_df[["division", "hh1"]].astype("string")
)
model_df["household_group"] = household_pairs.factorize(sort=True)[0].astype(np.int64)

X = model_df[PREDICTORS].copy()
y = model_df[TARGET].copy()
w = model_df[WEIGHT_COL].astype(float).copy()
groups = model_df["household_group"].to_numpy(dtype=np.int64)

print("Weight column:", WEIGHT_COL)
print("Rows retained:", len(model_df))
print("Unique division-households:", np.unique(groups).size)
print("Children per household (median/max):",
      model_df.groupby("household_group").size().median(),
      model_df.groupby("household_group").size().max())
print("Outcome counts:", y.value_counts().sort_index().to_dict())
print("Missing predictor cells:", int(X.isna().sum().sum()))

## 5. Household-aware held-out split

The first fold of a shuffled `StratifiedGroupKFold` is reserved as the test set. The remaining folds form the training set. This preserves approximate outcome balance while ensuring that household groups are disjoint. Because groups are indivisible, the test proportion may not be exactly 20%.

In [ ]:
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(outer_cv.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
w_train, w_test = w.iloc[train_idx].copy(), w.iloc[test_idx].copy()
g_train, g_test = groups[train_idx], groups[test_idx]

overlap = np.intersect1d(g_train, g_test)
assert overlap.size == 0, f"Household leakage detected: {overlap.size} groups overlap."
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError(
        "A grouped hold-out split contains only one outcome class. "
        "Try a different SEED or fewer outer folds; do not fall back to row-wise splitting."
    )

print(f"Train rows: {len(X_train):,} | households: {np.unique(g_train).size:,}")
print(f"Test rows:  {len(X_test):,} | households: {np.unique(g_test).size:,}")
print("Actual test fraction:", round(len(X_test) / len(X), 4))
print("Train prevalence:", round(y_train.mean(), 4), "| test prevalence:", round(y_test.mean(), 4))
print("Household overlap:", overlap.size)

## 6. MICE imputation, SMOTENC and post-resampling one-hot encoding

Each fold gets a fresh MICE kernel fitted on that fold's training observations only. `miceforest` uses chained conditional models and supports categorical predictors. Categorical columns are explicitly cast to pandas category dtype before fitting.

For the SMOTENC strategies, categorical columns remain categorical during resampling. One-hot encoding is deliberately delayed until after SMOTENC. For the weighted/no-SMOTE strategies, the same MICE and one-hot steps are used without resampling.

The child survey weight is not a class-balancing weight. In the SMOTENC strategies it is not passed to model fitting.

In [ ]:
def category_levels_from_training(Xf):
    """Learn category vocabularies from this training fold only."""
    levels = {}
    for col in CATEGORICAL_COLUMNS:
        observed = pd.Series(Xf[col].dropna().unique())
        if observed.empty:
            raise ValueError(
                f"{col} is entirely missing in a training fold; MICE cannot learn its categories."
            )
        levels[col] = sorted(observed.tolist(), key=lambda value: str(value))
    return levels

def cast_for_mice(frame, category_levels):
    out = frame[PREDICTORS].copy().reset_index(drop=True)
    for col in CATEGORICAL_COLUMNS:
        out[col] = pd.Categorical(out[col], categories=category_levels[col])
    for col in NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype(float)
    return out

def mice_impute_train_valid(Xf, Xv, iterations=MICE_ITERATIONS):
    """
    Fit MICE on this training fold only and impute its validation fold.

    Important safeguards:
    - Convert +/- infinity to missing before fitting.
    - Ask miceforest to model only columns that are actually missing in the
      training fold (not every predictor).
    - If a column is complete in training but missing in validation, use a
      training-only median/mode fallback; there is no fitted MICE model for it.
    - Try predictive mean matching first. If its KDTree receives non-finite
      donor predictions, retry without donor matching (mean_match_candidates=0).
    """
    levels = category_levels_from_training(Xf)
    train_mice = cast_for_mice(Xf, levels)
    valid_mice = cast_for_mice(Xv, levels)

    # A Stata special/malformed value or an upstream transform must never reach LightGBM/KDTree.
    for col in NUMERIC_COLUMNS:
        train_mice[col] = train_mice[col].replace([np.inf, -np.inf], np.nan)
        valid_mice[col] = valid_mice[col].replace([np.inf, -np.inf], np.nan)

    # MICE can only learn a conditional model when training data contain observed values.
    fallback_values = {}
    for col in PREDICTORS:
        if train_mice[col].notna().sum() == 0:
            raise ValueError(
                f"{col} is entirely missing in this training fold; "
                "it cannot be imputed without training observations."
            )
        if train_mice[col].isna().any():
            continue
        if valid_mice[col].isna().any():
            if col in CATEGORICAL_COLUMNS:
                mode = train_mice[col].mode(dropna=True)
                if mode.empty:
                    raise ValueError(f"No observed training category is available for {col}.")
                fallback_values[col] = mode.iloc[0]
            else:
                fallback_values[col] = float(train_mice[col].median())
            valid_mice[col] = valid_mice[col].fillna(fallback_values[col])

    imputation_variables = [
        col for col in PREDICTORS if train_mice[col].isna().any()
    ]
    if not imputation_variables:
        train_done, valid_done = train_mice.copy(), valid_mice.copy()
        kernel = None
    else:
        # Start with standard predictive mean matching (PMM).
        kernel = mf.ImputationKernel(
            data=train_mice,
            num_datasets=1,
            variable_schema=imputation_variables,
            mean_match_candidates=5,
            save_all_iterations_data=True,
            random_state=SEED
        )
        try:
            kernel.mice(iterations=iterations, verbose=False)
        except ValueError as exc:
            message = str(exc).lower()
            if "finite" not in message and "nan" not in message:
                raise
            warnings.warn(
                "MICE predictive mean matching produced non-finite donor predictions; "
                "restarting this fold with mean_match_candidates=0. "
                "This avoids the KDTree donor search while keeping the MICE chained models.",
                RuntimeWarning
            )
            kernel = mf.ImputationKernel(
                data=train_mice,
                num_datasets=1,
                variable_schema=imputation_variables,
                mean_match_candidates=0,
                save_all_iterations_data=True,
                random_state=SEED
            )
            kernel.mice(iterations=iterations, verbose=False)

        train_done = kernel.complete_data(dataset=0).reset_index(drop=True)
        valid_kernel = kernel.impute_new_data(
            new_data=valid_mice,
            datasets=[0],
            iterations=iterations,
            verbose=False,
            random_state=SEED
        )
        valid_done = valid_kernel.complete_data(dataset=0).reset_index(drop=True)

    for col in CATEGORICAL_COLUMNS:
        train_done[col] = pd.Categorical(train_done[col], categories=levels[col])
        valid_done[col] = pd.Categorical(valid_done[col], categories=levels[col])
    for col in NUMERIC_COLUMNS:
        train_done[col] = pd.to_numeric(train_done[col], errors="coerce").astype(float)
        valid_done[col] = pd.to_numeric(valid_done[col], errors="coerce").astype(float)

    # Final guard: never pass non-finite values to SMOTENC, the encoder, or a classifier.
    train_numeric = train_done[NUMERIC_COLUMNS].to_numpy(dtype=float)
    valid_numeric = valid_done[NUMERIC_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(train_numeric).all() or not np.isfinite(valid_numeric).all():
        bad_train = [
            c for c in NUMERIC_COLUMNS
            if not np.isfinite(train_done[c].to_numpy(dtype=float)).all()
        ]
        bad_valid = [
            c for c in NUMERIC_COLUMNS
            if not np.isfinite(valid_done[c].to_numpy(dtype=float)).all()
        ]
        raise ValueError(
            f"MICE produced non-finite numeric values. "
            f"Train columns={bad_train}; validation columns={bad_valid}"
        )
    if train_done.isna().any().any() or valid_done.isna().any().any():
        bad_train = train_done.columns[train_done.isna().any()].tolist()
        bad_valid = valid_done.columns[valid_done.isna().any()].tolist()
        raise ValueError(
            f"MICE left missing values. Train columns={bad_train}; validation columns={bad_valid}"
        )
    return train_done, valid_done, kernel, levels

def post_smote_onehot(Xfit, Xvalid, yfit, use_smote):
    yfit_array = np.asarray(yfit, dtype=int)
    if use_smote:
        cat_idx = [Xfit.columns.get_loc(c) for c in CATEGORICAL_COLUMNS]
        counts = pd.Series(yfit_array).value_counts()
        if len(counts) != 2 or counts.min() < 2:
            raise ValueError(
                f"SMOTENC needs at least two observations per class in each training fold; "
                f"found {counts.to_dict()}."
            )
        sampler = SMOTENC(
            categorical_features=cat_idx,
            random_state=SEED,
            k_neighbors=min(5, int(counts.min()) - 1)
        )
        X_res, y_res = sampler.fit_resample(Xfit, yfit_array)
        Xfit = pd.DataFrame(X_res, columns=Xfit.columns)
        for col in CATEGORICAL_COLUMNS:
            Xfit[col] = pd.Categorical(Xfit[col], categories=Xfit[col].dropna().unique())
        yfit_array = np.asarray(y_res, dtype=int)

    # One-hot encoding is intentionally fitted only after SMOTENC.
    encoder = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
            ("numeric", "passthrough", NUMERIC_COLUMNS)
        ],
        remainder="drop"
    )
    Xfit_enc = encoder.fit_transform(Xfit)
    Xvalid_enc = encoder.transform(Xvalid)
    feature_names = encoder.get_feature_names_out()
    return Xfit_enc, Xvalid_enc, yfit_array, encoder, feature_names

MODEL_PARAMS = {
    "XGBoost": {
        "n_estimators": 700, "max_depth": 5, "learning_rate": 0.05,
        "min_child_weight": 3, "subsample": 0.85, "colsample_bytree": 0.85,
        "reg_lambda": 5.0
    },
    "LightGBM": {
        "n_estimators": 700, "num_leaves": 31, "max_depth": -1,
        "learning_rate": 0.05, "min_child_samples": 25,
        "subsample": 0.85, "subsample_freq": 1,
        "colsample_bytree": 0.85, "reg_lambda": 5.0
    }
}

def make_estimator(model_name):
    if model_name == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=SEED,
            verbosity=0, **MODEL_PARAMS[model_name]
        )
    return LGBMClassifier(
        objective="binary", n_jobs=-1, random_state=SEED,
        verbosity=-1, **MODEL_PARAMS[model_name]
    )

def fit_configuration(model_name, strategy, Xf, yf, wf, Xv):
    use_smote = strategy == "SMOTENC_no_ch_weight"
    use_weights = strategy == "ch_weight_no_SMOTENC"
    if not (len(Xf) == len(yf) == len(wf)):
        raise ValueError("Training predictors, outcome and child weights are misaligned.")

    Xf_imp, Xv_imp, mice_kernel, levels = mice_impute_train_valid(Xf, Xv)
    Xf_enc, Xv_enc, yfit, encoder, feature_names = post_smote_onehot(
        Xf_imp, Xv_imp, yf, use_smote
    )
    estimator = make_estimator(model_name)
    if use_weights:
        fit_weights = np.asarray(wf, dtype=float)
        fit_weights = fit_weights / fit_weights.mean()
        estimator.fit(Xf_enc, yfit, sample_weight=fit_weights)
    else:
        estimator.fit(Xf_enc, yfit)

    return {
        "model": estimator, "encoder": encoder, "mice_kernel": mice_kernel,
        "category_levels": levels, "feature_names": feature_names,
        "X_train_imputed": Xf_imp, "X_valid_imputed": Xv_imp,
        "X_train_encoded": Xf_enc, "X_valid_encoded": Xv_enc
    }

def predict_bundle(bundle, Xnew):
    new_mice = cast_for_mice(Xnew, bundle["category_levels"])
    new_kernel = bundle["mice_kernel"].impute_new_data(
        new_data=new_mice, datasets=[0], iterations=MICE_ITERATIONS,
        verbose=False, random_state=SEED
    )
    new_done = new_kernel.complete_data(dataset=0).reset_index(drop=True)
    for col in CATEGORICAL_COLUMNS:
        new_done[col] = pd.Categorical(
            new_done[col], categories=bundle["category_levels"][col]
        )
    for col in NUMERIC_COLUMNS:
        new_done[col] = pd.to_numeric(new_done[col], errors="coerce").astype(float)
    if new_done.isna().any().any():
        raise ValueError("MICE prediction-time imputation left missing values.")
    encoded = bundle["encoder"].transform(new_done)
    return bundle["model"].predict_proba(encoded)[:, 1]

## 7. Four model configurations with household-aware cross-validation

Every fold is split by household groups. MICE is refitted within each fold. SMOTENC is applied only to the training part of a fold and never to its validation part. The threshold is selected from training-only out-of-fold predictions; no held-out test labels are used to choose it.

In [ ]:
STRATEGIES = {
    "SMOTENC_no_ch_weight": {"smote": True, "weights": False},
    "ch_weight_no_SMOTENC": {"smote": False, "weights": True}
}
CONFIGS = [(model_name, strategy) for strategy in STRATEGIES for model_name in MODEL_PARAMS]

inner_cv = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
oof_probabilities, selected_thresholds = {}, {}
cv_rows = []

for model_name, strategy in CONFIGS:
    key = f"{model_name} | {strategy}"
    oof = np.full(len(X_train), np.nan, dtype=float)
    weighted_objective = STRATEGIES[strategy]["weights"]

    for fold, (tr, va) in enumerate(
        inner_cv.split(X_train, y_train, groups=g_train), start=1
    ):
        fold_overlap = np.intersect1d(g_train[tr], g_train[va])
        assert fold_overlap.size == 0, (
            f"Household leakage in {key}, fold {fold}: {fold_overlap.size} overlapping groups."
        )
        if y_train.iloc[tr].nunique() != 2 or y_train.iloc[va].nunique() != 2:
            raise ValueError(
                f"{key}, fold {fold} has only one class in train/validation. "
                "Reduce CV_FOLDS or change SEED; do not use row-wise CV."
            )

        bundle = fit_configuration(
            model_name, strategy,
            X_train.iloc[tr], y_train.iloc[tr], w_train.iloc[tr],
            X_train.iloc[va]
        )
        oof[va] = bundle["model"].predict_proba(bundle["X_valid_encoded"])[:, 1]
        print(key, "| fold", fold, "complete")

    if np.isnan(oof).any():
        raise RuntimeError(f"Missing OOF predictions for {key}.")
    oof_probabilities[key] = oof
    threshold_weight = w_train.to_numpy() if weighted_objective else None
    precision, recall, thresholds = precision_recall_curve(
        y_train, oof, sample_weight=threshold_weight
    )
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best = int(np.nanargmax(f1_values))
    selected_thresholds[key] = float(thresholds[best]) if len(thresholds) else 0.5

    cv_rows.append({
        "model": model_name, "strategy": strategy,
        "threshold": selected_thresholds[key],
        "training_oof_f1": float(f1_values[best]),
        "training_oof_roc_auc": roc_auc_score(y_train, oof, sample_weight=threshold_weight),
        "training_oof_pr_auc": average_precision_score(y_train, oof, sample_weight=threshold_weight)
    })

cv_results = pd.DataFrame(cv_rows)
display(cv_results.round(4))

## 8. Held-out evaluation: metrics, confusion matrices and ROC-AUC

The test set contains households never used for model fitting, imputation, resampling, threshold selection or cross-validation. Confusion matrices show child counts (unweighted); weighted and unweighted metrics are both reported.

In [ ]:
test_probabilities, test_predictions, heldout_bundles = {}, {}, {}
metric_rows = []
for model_name, strategy in CONFIGS:
    key = f"{model_name} | {strategy}"
    bundle = fit_configuration(
        model_name, strategy, X_train, y_train, w_train, X_test
    )
    probability = bundle["model"].predict_proba(bundle["X_valid_encoded"])[:, 1]
    prediction = (probability >= selected_thresholds[key]).astype(int)
    heldout_bundles[key] = bundle
    test_probabilities[key] = probability
    test_predictions[key] = prediction

    metric_rows.append({
        "model": model_name, "strategy": strategy,
        "threshold": selected_thresholds[key],
        "weighted_precision": precision_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_recall": recall_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_f1": f1_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_roc_auc": roc_auc_score(y_test, probability, sample_weight=w_test),
        "weighted_pr_auc": average_precision_score(y_test, probability, sample_weight=w_test),
        "unweighted_precision": precision_score(y_test, prediction, zero_division=0),
        "unweighted_recall": recall_score(y_test, prediction, zero_division=0),
        "unweighted_f1": f1_score(y_test, prediction, zero_division=0),
        "unweighted_roc_auc": roc_auc_score(y_test, probability),
        "unweighted_pr_auc": average_precision_score(y_test, probability),
        "accuracy": accuracy_score(y_test, prediction)
    })

heldout_results = pd.DataFrame(metric_rows)
display(heldout_results.sort_values("weighted_f1", ascending=False).round(4))

In [ ]:
# Confusion matrices: raw child counts and survey-weighted totals.
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (key, pred) in zip(axes.ravel(), test_predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    ConfusionMatrixDisplay(
        cm, display_labels=["No wasting", "Any wasting"]
    ).plot(ax=ax, cmap="Blues", colorbar=False, values_format=",")
    ax.set_title(key)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Observed")
fig.suptitle("Held-out confusion matrices (unweighted child counts)", fontsize=14)
plt.tight_layout()
plt.show()

cm_rows = []
for key, pred in test_predictions.items():
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    wtn, wfp, wfn, wtp = confusion_matrix(
        y_test, pred, labels=[0, 1], sample_weight=w_test
    ).ravel()
    cm_rows.append({
        "model_configuration": key,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "weighted_TN": float(wtn), "weighted_FP": float(wfp),
        "weighted_FN": float(wfn), "weighted_TP": float(wtp)
    })
confusion_results = pd.DataFrame(cm_rows)
display(confusion_results)

In [ ]:
# ROC curves: solid = unweighted, dashed = weighted by child sample weights.
fig, ax = plt.subplots(figsize=(10, 8))
for key, prob in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_u = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, label=f"{key} | unweighted AUC={auc_u:.3f}")
    fpr_w, tpr_w, _ = roc_curve(y_test, prob, sample_weight=w_test)
    auc_w = roc_auc_score(y_test, prob, sample_weight=w_test)
    ax.plot(fpr_w, tpr_w, linestyle="--", label=f"{key} | weighted AUC={auc_w:.3f}")
ax.plot([0, 1], [0, 1], "k:", label="No-discrimination reference")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="Household-held-out ROC curves")
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Explainability: SHAP, built-in importance and permutation importance

SHAP is computed on a sample of the untouched held-out observations. The one-hot feature names are retained so the plots identify the encoded predictor levels. Built-in gain/split importance and permutation importance provide complementary summaries. Permutation importance is calculated on held-out data and can be computationally expensive; reduce `n_repeats` or sample size if needed.

These are predictive explanations, not causal effects.

In [ ]:
# SHAP explanations on held-out observations already imputed with the training-fitted MICE kernel.
SHAP_SAMPLE = min(400, len(X_test))
shap_rng = np.random.RandomState(SEED)
shap_idx = shap_rng.choice(len(X_test), size=SHAP_SAMPLE, replace=False)
explainability_rows = []

for key, bundle in heldout_bundles.items():
    model = bundle["model"]
    encoded = bundle["X_valid_encoded"][shap_idx]
    encoded_dense = encoded.toarray() if hasattr(encoded, "toarray") else np.asarray(encoded)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(encoded_dense)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    print("\nSHAP summary:", key)
    shap.summary_plot(
        shap_values, encoded_dense, feature_names=bundle["feature_names"],
        max_display=20, show=True
    )
    shap.summary_plot(
        shap_values, encoded_dense, feature_names=bundle["feature_names"],
        plot_type="bar", max_display=20, show=True
    )

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    for name, value in zip(bundle["feature_names"], mean_abs_shap):
        explainability_rows.append({
            "model_configuration": key,
            "feature": name,
            "mean_abs_shap": float(value)
        })

    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
        top_idx = np.argsort(importance)[::-1][:20]
        plt.figure(figsize=(9, 6))
        plt.barh(
            np.array(bundle["feature_names"])[top_idx][::-1],
            np.array(importance)[top_idx][::-1]
        )
        plt.title(f"Built-in feature importance — {key}")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()

shap_importance = pd.DataFrame(explainability_rows)
display(
    shap_importance.sort_values(
        ["model_configuration", "mean_abs_shap"], ascending=[True, False]
    ).groupby("model_configuration").head(20)
)

In [ ]:
# Permutation importance on held-out encoded observations.
# It is complementary to SHAP and measures the drop in ROC-AUC after shuffling each encoded feature.
perm_rows = []
for key, bundle in heldout_bundles.items():
    Xenc = bundle["X_valid_encoded"]
    result = permutation_importance(
        bundle["model"], Xenc, y_test, scoring="roc_auc",
        n_repeats=5, random_state=SEED, n_jobs=-1
    )
    names = bundle["feature_names"]
    top = np.argsort(result.importances_mean)[::-1][:20]
    for j in top:
        perm_rows.append({
            "model_configuration": key,
            "feature": names[j],
            "permutation_auc_decrease": float(result.importances_mean[j]),
            "permutation_std": float(result.importances_std[j])
        })

permutation_results = pd.DataFrame(perm_rows)
display(permutation_results.sort_values(
    ["model_configuration", "permutation_auc_decrease"],
    ascending=[True, False]
))

## 10. Automated comparison discussion and export

The discussion below is generated from the actual held-out results after the notebook is run. Do not treat it as a result before execution.

In [ ]:
best_wf1 = heldout_results.loc[heldout_results["weighted_f1"].idxmax()]
best_auc = heldout_results.loc[heldout_results["weighted_roc_auc"].idxmax()]
print("RESULTS DISCUSSION")
print("-" * 60)
print(f"Highest weighted F1: {best_wf1['model']} / {best_wf1['strategy']} "
      f"(F1={best_wf1['weighted_f1']:.4f}; threshold={best_wf1['threshold']:.3f}).")
print(f"Highest weighted ROC-AUC: {best_auc['model']} / {best_auc['strategy']} "
      f"(ROC-AUC={best_auc['weighted_roc_auc']:.4f}).")
for _, row in heldout_results.iterrows():
    print(f"- {row['model']} | {row['strategy']}: "
          f"weighted P/R/F1={row['weighted_precision']:.3f}/"
          f"{row['weighted_recall']:.3f}/{row['weighted_f1']:.3f}; "
          f"weighted ROC-AUC={row['weighted_roc_auc']:.3f}; "
          f"unweighted ROC-AUC={row['unweighted_roc_auc']:.3f}.")
print("\nInterpretation guide:")
print("* Higher recall means fewer children with wasting are missed, often at the cost of more false positives.")
print("* Confusion matrices expose false negatives and false positives at the selected training-only threshold.")
print("* ROC-AUC measures ranking across thresholds; PR-AUC is especially informative when the positive class is uncommon.")
print("* Weighted metrics incorporate child sampling weights; unweighted metrics describe this analytic sample.")
print("* Household grouping prevents sibling/household leakage but does not itself account for every MICS survey-design feature.")

OUTPUT = Path("outputs") / "any_wasting_household_aware_mice"
OUTPUT.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(OUTPUT / "grouped_cross_validation.csv", index=False)
heldout_results.to_csv(OUTPUT / "heldout_metrics.csv", index=False)
confusion_results.to_csv(OUTPUT / "heldout_confusion_matrices.csv", index=False)
shap_importance.to_csv(OUTPUT / "shap_feature_importance.csv", index=False)
permutation_results.to_csv(OUTPUT / "permutation_feature_importance.csv", index=False)
print("Saved outputs to:", OUTPUT.resolve())

## 11. Methodological notes and interpretation

1. **Household dependence:** the grouping key is the exact pair `(division, hh1)`. Both the held-out split and every CV fold use `StratifiedGroupKFold`; the code asserts that train and validation household IDs are disjoint. This avoids sibling/household leakage.
2. **MICE:** a new `miceforest` kernel is fitted on each training fold only. Its conditional models impute that fold's validation rows. The held-out test set is imputed from a kernel fitted on the complete training split. Category vocabularies are learned from each training fold, not from the full dataset.
3. **SMOTENC and encoding:** SMOTENC is applied only to training data, with categorical columns identified as categorical. One-hot encoding is fitted only after resampling. Validation/test rows are never oversampled.
4. **Survey weights:** `ch_weight` is preferred, with `chweight` as a fallback. The SMOTENC strategy does not use child weights in fitting; the weighted strategy uses normalized child weights and no SMOTENC. Weighted and unweighted evaluation metrics are both reported.
5. **Confusion matrices:** raw counts are shown, along with weighted TN/FP/FN/TP totals. TN = correctly identified non-wasting children; FP = non-wasting children flagged as wasting; FN = wasting children missed; TP = correctly identified wasting children.
6. **ROC-AUC and PR-AUC:** both weighted and unweighted ROC-AUC/PR-AUC are calculated. PR-AUC is particularly informative when the positive class is uncommon.
7. **Explainability:** SHAP beeswarm and bar summaries, native tree importance, and held-out permutation importance are provided. These explain model predictions; they do not establish causal effects.
8. **No fabricated results:** run all cells from top to bottom using the revised MICS data. Interpret generated tables and plots only after successful execution.

**Reproducibility caution:** MICE, cross-validation, tree fitting and SHAP may take substantial time. For a quick smoke test, reduce `CV_FOLDS`, `MICE_ITERATIONS`, estimator counts and `SHAP_SAMPLE`; restore the planned values for the final run.